In [3]:
import keras_hub
import keras
import tensorflow as tf
import os


2025-05-30 06:17:43.569729: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-30 06:17:43.710699: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-30 06:17:43.873597: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1748582264.018230   22257 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1748582264.062444   22257 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-05-30 06:17:44.362394: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU ins

# IMDB movie reviews dataset

In [5]:
import pandas as pd
x_trainVal = pd.read_csv('imdb_x_trainVal.csv')
y_trainVal = pd.read_csv('imdb_y_trainVal.csv')
x_trainValrev=x_trainVal['review'].values
labels_trainVal = y_trainVal['sentiment'].values

In [6]:
# first movie review:
x_trainValrev[0]

'In spite of sterling work by the supporting actors, and an intelligent script by Alan Plater, this film suffers from a fatal flaw - the lack of charm of the central character/actor. One of the characters describes Richard E Grant\'s character as "a whining little turd" and unfortunately this sums him up perfectly. There is nothing about him or his performance to make it credible that his girlfriend and upper-class publisher/friend would spend so much time and emotional effort on him. He is rude, arrogant, selfish, self-destructive and thoroughly annoying. The part called for an actor who can make you love him even when he is being a prate - a Ewan McGregor, for example.<br /><br />All of the witty satire on the class system etc was wasted, thanks to this irritating and thoroughly unlikeable performance. All I wanted to do was shake him and tell him to get over himself.'

## We create a ***tokenizer*** using the words of the IMDB dataset

In [13]:
MAX_SEQUENCE_LENGTH = 512
VOCAB_SIZE = 15000
BATCH_SIZE = 64

In [16]:
aux = tf.data.Dataset.from_tensor_slices(x_trainValrev[:1000])
aux = aux.batch(BATCH_SIZE) 
vocab =  keras_hub.tokenizers.compute_word_piece_vocabulary(aux ,vocabulary_size=VOCAB_SIZE)

2025-05-30 06:21:53.808362: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [20]:
# some tokens
vocab[1100:1110]

['comic',
 'crew',
 'dumb',
 'except',
 'hate',
 'knew',
 'laughing',
 'nearly',
 'remake',
 'songs']

In [17]:
tokenizer = keras_hub.tokenizers.WordPieceTokenizer(
    vocabulary=vocab,
    lowercase=False,
    sequence_length=MAX_SEQUENCE_LENGTH,
)

In [18]:
tokensOrds=tokenizer.tokenize(x_trainValrev[0])
#tokensOrds

In [38]:
tokens =[vocab[i] for i in tokensOrds]
#tokens

In [37]:
EMBED_DIM = 128
INTERMEDIATE_DIM = 512
EPOCHS = 3

In [40]:
def g(x,y):
    return tokenizer(x)[:MAX_SEQUENCE_LENGTH],y

In [49]:
dataset = tf.data.Dataset.from_tensor_slices((x_trainValrev,labels_trainVal) )
dataset = dataset.map(g)
dataset = dataset.padded_batch(8,padded_shapes=([MAX_SEQUENCE_LENGTH],[]))
it = iter(dataset)
ex = next(it)

In [50]:
print(ex)

(<tf.Tensor: shape=(8, 512), dtype=int32, numpy=
array([[ 257, 3845,  116, ...,    0,    0,    0],
       [  43,  705,  367, ...,    0,    0,    0],
       [1603,  571,  165, ...,    0,    0,    0],
       ...,
       [ 721,  403,  165, ...,  819, 1435, 3452],
       [  45, 2830,  124, ...,    0,    0,    0],
       [  45,  941,  145, ...,    0,    0,    0]], dtype=int32)>, <tf.Tensor: shape=(8,), dtype=int64, numpy=array([0, 1, 0, 0, 0, 0, 0, 1])>)


# Build a transformer network (encoder) to classify IMDB reviews

In [47]:
inputs_tokenIds = tf.keras.Input(shape=(MAX_SEQUENCE_LENGTH,),name='token_ids')
emBedding = keras.layers.Embedding(input_dim = VOCAB_SIZE,output_dim=16)
emB = emBedding(inputs_tokenIds)
posEmb = keras_hub.layers.PositionEmbedding(MAX_SEQUENCE_LENGTH)
x1 = posEmb(emB) + emB
encBlock1 =keras_hub.layers.TransformerEncoder(intermediate_dim=16, num_heads=4,dropout=0)
x2 = encBlock1(x1)
out = tf.keras.layers.Dense(1, activation='sigmoid')(x2[:,-1])
classifModel = tf.keras.Model(inputs=inputs_tokenIds, outputs=out)

In [51]:
classifModel(ex[0])

<tf.Tensor: shape=(8, 1), dtype=float32, numpy=
array([[0.19668594],
       [0.19539228],
       [0.19425568],
       [0.19267087],
       [0.6914681 ],
       [0.58674103],
       [0.19429667],
       [0.19268651]], dtype=float32)>

In [52]:
classifModel.compile(loss='binary_crossentropy', optimizer=tf.keras.optimizers.Adam(0.0001), metrics=['accuracy'])

In [53]:
classifModel.fit(dataset, epochs=2)

Epoch 1/2
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 308s 61ms/step - accuracy: 0.5995 - loss: 0.6516
Epoch 2/2
5000/5000 ━━━━━━━━━━━━━━━━━━━━ 307s 61ms/step - accuracy: 0.8548 - loss: 0.3383
